# 10. Train-Test Split — Assembling the Real System, Honestly Evaluated

**Building a Heart Disease Risk-Screening System — Notebook 10 of 12, Stage 5: Building and Validating the Predictive Core**

Notebook 9 proved the regression mechanics on a toy relationship. This notebook
assembles the *actual* system this whole module has been building toward: a model
predicting `target` (disease presence) from the inputs validated in Stage 4 —
and, critically, evaluates it the way it will actually be used: on patients it
hasn't seen.

## The topic

Evaluating a model on the same data it was fit on is optimistic by construction —
the model had every opportunity to fit quirks specific to those exact patients. A
**train-test split** holds out data the model never learns from, so its reported
performance reflects how it will do on new patients, not how well it memorized
old ones.

## Why it matters for this system

This is the last checkpoint before the system could plausibly be trusted with a
new patient. Skip it, and the "90% accurate" the team reports internally could be
substantially worse in the clinic — the exact gap this notebook exists to expose
before it becomes a real-world surprise.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
inputs = ["age", "sex", "cp", "trestbps", "chol", "thalach", "exang"]  # Stage 4's validated inputs
X = df[inputs]
y = df["target"]
print(X.shape)

## The toolkit

| Tool | Use |
|---|---|
| **Train/test split** | The baseline honest-evaluation setup |
| **K-fold cross-validation** | More stable estimate than one split, using every row for both roles across folds |
| **Time-based split** | When new patients enter the registry over time and future leakage into training is a risk |
| **Leakage checks** | Catching preprocessing or feature-derivation mistakes a split alone won't catch |

## How to choose

Use a single train/test split for a quick, interpretable check and for the final
reportable number. Use k-fold cross-validation when you want a more stable estimate
and can afford the extra compute — especially valuable on a registry this size
(438 patients), where a single 20% test set is under 90 patients and could itself
be an unlucky or lucky draw. Reach for a time-based split specifically if patients
are added to the registry over time and a random split could let the model
implicitly "see the future." Leakage checks aren't a method you choose between —
run them regardless of which split strategy you use.

## Applied to the registry

### The problem: evaluating on training data is optimistic

In [ ]:
model = LogisticRegression(max_iter=2000).fit(X, y)
train_predictions = model.predict_proba(X)[:, 1]
train_auc = roc_auc_score(y, train_predictions)
print(f"AUC on TRAINING data: {train_auc:.3f}")
print("This number answers 'how well does the model describe patients it already saw,'")
print("not 'how well will it screen the next patient who walks in.'")

### The fix: hold out data the model never sees

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)} patients   Test: {len(X_test)} patients")

model_split = LogisticRegression(max_iter=2000).fit(X_train, y_train)
test_auc = roc_auc_score(y_test, model_split.predict_proba(X_test)[:, 1])
test_acc = accuracy_score(y_test, model_split.predict(X_test))

print(f"\nAUC on TEST data:  {test_auc:.3f}   (training AUC was {train_auc:.3f})")
print(f"Accuracy on TEST data: {test_acc:.3f}")

The gap between training and test AUC is the honest cost of Notebook 11's
overfitting risk, made visible here first. A small gap is reassuring; a large one
is the alarm Notebook 11 investigates directly.

### Leakage trap: preprocessing fit on the whole dataset before splitting

In [ ]:
from sklearn.preprocessing import StandardScaler

# WRONG: fit the scaler on everything, THEN split
scaler_leaky = StandardScaler().fit(X)
X_scaled_leaky = pd.DataFrame(scaler_leaky.transform(X), columns=X.columns)
X_train_lk, X_test_lk, y_train_lk, y_test_lk = train_test_split(X_scaled_leaky, y, test_size=0.2, random_state=42, stratify=y)

# RIGHT: split first, fit the scaler on training data ONLY
X_train_ok, X_test_ok, y_train_ok, y_test_ok = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler_ok = StandardScaler().fit(X_train_ok)
X_train_scaled = scaler_ok.transform(X_train_ok)
X_test_scaled = scaler_ok.transform(X_test_ok)

model_leaky = LogisticRegression(max_iter=2000).fit(X_train_lk, y_train_lk)
model_ok = LogisticRegression(max_iter=2000).fit(X_train_scaled, y_train_ok)

print(f"AUC with leaky scaling:   {roc_auc_score(y_test_lk, model_leaky.predict_proba(X_test_lk)[:,1]):.4f}")
print(f"AUC with correct scaling: {roc_auc_score(y_test_ok, model_ok.predict_proba(X_test_scaled)[:,1]):.4f}")
print("\nThe difference is small here for a StandardScaler, but the same mistake with a")
print("target-derived feature (e.g. mean-encoding a category by disease rate) can inflate")
print("reported performance dramatically -- always fit preprocessing on training data only.")

### K-fold cross-validation: one split can be an unlucky draw

With only ~88 patients in a single test split, cross-validation gives a far more
stable picture.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=kf, scoring="roc_auc")
print("AUC across 5 folds:", np.round(cv_scores, 3))
print(f"Mean: {cv_scores.mean():.3f}   Std: {cv_scores.std():.3f}")
print(f"\nSingle split gave AUC={test_auc:.3f} -- see where that falls within this spread.")

## Systems view — what this stage hands to the next one

The system now has an honest, cross-validated performance number instead of an
optimistic training-data one. Notebook 11 asks *why* the training/test gap exists
when it's large, and Notebook 12 formalizes the tradeoff behind choosing how
complex a model to trust.

## Try it yourself

1. Repeat the split with `test_size=0.35` and `test_size=0.1` — how much does the
   AUC estimate's *stability* change with test set size on a registry this small?
2. Add two more validated inputs from Stage 4 (`slope`, `ca`) to `inputs` and
   re-run the cross-validation — does AUC improve, and does the fold-to-fold spread
   change?
3. Engineer a deliberately leaky feature (e.g. a column derived directly from
   `target` with noise added, as in Notebook's leakage discussion) and confirm it
   produces a suspiciously large AUC jump — the same red flag pattern to watch for
   with any new feature added to this system later.